# Manual Inspection: Rescore Results

Browse samples marked as **wrong** (`score_xverify == 0`) from the latest rescoring run to verify whether the verifier is correct.

In [8]:
import pandas as pd
import random
from IPython.display import display, HTML, Markdown

RESCORE_PATH = "../data/results/rescore_7b_all8_fixed.parquet"

df = pd.read_parquet(RESCORE_PATH)
print(f"Total rows: {len(df)}")
print(f"score_xverify value counts:")
print(df['score_xverify'].value_counts())
print(f"\nSources:")
print(df['source'].value_counts())

Total rows: 6866
score_xverify value counts:
score_xverify
0.0    4141
1.0    2725
Name: count, dtype: int64

Sources:
source
UGPhysics        5451
PHYSICS           805
SciBench_RL       280
OlympiadBench     230
PHYBench          100
Name: count, dtype: int64


In [9]:
# --- Filters ---
# Change these to slice the population you want to inspect

VERDICT     = 0          # 0 = wrong, 1 = correct, None = both
SOURCE      = None       # e.g. 'OlympiadBench', 'PHYSICS', None = all
ANSWER_TYPE = None       # e.g. 'numerical', 'expression', None = all
TRUNCATED   = False      # False = non-truncated only, True = truncated only, None = both
N_SAMPLE    = 20         # how many rows to sample
SEED        = 42

mask = pd.Series([True] * len(df), index=df.index)
if VERDICT is not None:
    mask &= df['score_xverify'] == VERDICT
if SOURCE is not None:
    mask &= df['source'] == SOURCE
if ANSWER_TYPE is not None:
    mask &= df['answer_type'].str.contains(ANSWER_TYPE, na=False)
if TRUNCATED is not None:
    mask &= df['truncated'] == TRUNCATED

pool = df[mask]
sample = pool.sample(min(N_SAMPLE, len(pool)), random_state=SEED).reset_index(drop=True)
print(f"Pool size: {len(pool)}  |  Sampled: {len(sample)}")

Pool size: 2738  |  Sampled: 20


In [10]:
def render_row(i, row):
  verdict_color = "#d9534f" if row['score_xverify'] == 0 else "#5cb85c"
  verdict_label = "WRONG" if row['score_xverify'] == 0 else "CORRECT"
  trunc_warn = " <b>[TRUNCATED]</b>" if row.get('truncated') else ""

  html = f"""
  <div style='border:2px solid {verdict_color}; border-radius:6px; padding:14px; margin-bottom:20px; font-family:monospace; color:black'>
    <div style='display:flex; justify-content:space-between; margin-bottom:8px'>
    <b style='font-size:1.1em'>#{i+1} &nbsp; {row['problem_id']}</b>
    <span style='color:black; font-weight:bold'>{verdict_label}{trunc_warn}</span>
    </div>
    <div style='margin-bottom:4px'><b>Source:</b> {row['source']} &nbsp;|&nbsp; <b>Answer type:</b> {row['answer_type']}</div>
    <hr style='margin:8px 0'/>

    <div style='background:#f8f8f8; padding:8px; border-radius:4px; margin-bottom:8px; white-space:pre-wrap'>
    <b>Question:</b><br>{row['problem']}
    </div>

    <div style='background:#fff3cd; padding:8px; border-radius:4px; margin-bottom:8px; white-space:pre-wrap'>
    <b>Gold answer:</b> {row['gold_answer']}
    {("<br><b>Unit:</b> " + str(row['unit'])) if row.get('unit') else ""}
    {("<br><b>Tolerance:</b> " + str(row['tolerance'])) if row.get('tolerance') else ""}
    </div>

    <div style='background:#e8f4fd; padding:8px; border-radius:4px; margin-bottom:8px; white-space:pre-wrap'>
    <b>Model answer (pred_text):</b><br>{row['pred_text']}
    </div>

    <details>
    <summary style='cursor:pointer; color:#555'>Raw output (click to expand)</summary>
    <div style='background:#f0f0f0; padding:8px; border-radius:4px; white-space:pre-wrap; max-height:400px; overflow-y:auto'>{row['raw_output']}</div>
    </details>
  </div>
  """
  return html

for i, row in sample.iterrows():
  display(HTML(render_row(i, row)))

is UGPhysics_QuantumMechanics_00938 gold answer wrong? it doesn match question 
  also such as UGPhysics_QuantumMechanics_00954 why it has no units                                                                                                                                         
  UGPhysics_Relativity_00149 unit is weird                                                                                                                                                                  
  also is UGPhysics_QuantumMechanics_00469 the equivalent results but non-standard unit?                                                                                                                    
  UGPhysics_TheoreticalMechanics_00091 answer seems off as well                                                                                                                                             
  PHYSICS_00083 seems like an open-ended question (the expression notations are not defined)                                                                                                                
  PHYSICS_01413 has 2 questions but gold answer only 1 answer                                                                                                                                               
  UGPhysics_ClassicalMechanics_00447 also seems semi open ended without proper notation definition and is the result actually equivalent?                                                                   
  UGPhysics_SemiconductorPhysics_00116 seems also equivalent                                                                                                                                                
  UGPhysics_Relativity_00199 is the answer really 0? gold says 0 but it is strange                                                                                                                          
  UGPhysics_Thermodynamics_00195 also didnt define M                                                                                                                                                        
  UGPhysics_AtomicPhysics_00408 also seems ambiguous about what it wants (explanation or just that example?)    

In [11]:
# --- Jump to a specific problem_id ---
PROBLEM_ID = ""  # fill in a problem_id string to inspect it directly

if PROBLEM_ID:
    rows = df[df['problem_id'] == PROBLEM_ID]
    if rows.empty:
        print(f"Not found: {PROBLEM_ID}")
    else:
        for i, row in rows.iterrows():
            display(HTML(render_row(0, row)))

In [12]:
# --- Summary stats by source and verdict ---
summary = (
    df.groupby(['source', 'score_xverify'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0.0: 'wrong', 1.0: 'correct'})
)
summary['total'] = summary.sum(axis=1)
summary['acc'] = (summary.get('correct', 0) / summary['total'] * 100).round(1)
display(summary)

score_xverify,wrong,correct,total,acc
source,,,,
OlympiadBench,165,65,230,28.3
PHYBench,88,12,100,12.0
PHYSICS,482,323,805,40.1
SciBench_RL,49,231,280,82.5
UGPhysics,3357,2094,5451,38.4
